# Every LLM Has a Latent World Model Inside
## Full Pipeline: D0 + D1 + D2

**Runtime**: Seleziona **GPU** da `Runtime > Change runtime type > T4 GPU` (o meglio, A100 se disponibile).

**Tempo stimato**:
- D0 + D1 (3 geometrie ciascuno): ~15-20 min su T4
- D2 encoding: ~10-15 min (LM scoring è la parte lenta)
- D2 training: ~20-30 min

Puoi runnare le sezioni indipendentemente — D0/D1 non richiedono D2 e viceversa.

## 0. Setup

In [ ]:
# Clona il repo (idempotente: se esiste già fa git pull)
import os
REPO = "Every_LLM_has_a_Latent_World_Model_inside"
if not os.path.isdir(REPO):
    !git clone https://github.com/PiGrieco/{REPO}.git
else:
    !cd {REPO} && git pull --ff-only || true
%cd {REPO}

In [ ]:
# Installa dipendenze
# NOTE: NON reinstalliamo torch: Colab ha già una build compatibile con la sua CUDA.
# Installiamo solo ciò che manca / è più recente rispetto all'immagine Colab.
!pip install -q -U transformers sentence-transformers datasets \
    faiss-cpu pyyaml umap-learn

# In alternativa (se vuoi i vincoli esatti del repo, più lento):
# !pip install -q -r requirements.txt

In [ ]:
# Verifica GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 1. D0 — Synthetic Time-Reversal (H1 + H2)

Il test fondamentale: il metric Lorentziano distingue traiettorie forward da reversed?

Runna tutte e 3 le geometrie (Lorentzian, Riemannian, Euclidean) per confronto.

In [ ]:
!python -m scripts.train --config configs/d0_synthetic.yaml --geometry lorentzian

In [ ]:
!python -m scripts.train --config configs/d0_synthetic.yaml --geometry riemannian

In [ ]:
!python -m scripts.train --config configs/d0_synthetic.yaml --geometry euclidean

In [ ]:
# Confronta risultati D0
import json, os

def _fmt(v, w=10, prec=4):
    """Format numerici con .4f, stringhe lasciate intatte (evita ValueError su N/A)."""
    if isinstance(v, (int, float)):
        return f"{v:<{w}.{prec}f}"
    return f"{str(v):<{w}}"

print("=" * 70)
print("D0 RESULTS — TIME-REVERSAL EXPERIMENT")
print("=" * 70)
print(f"{'Geometry':<15} {'M1 (timelike)':<18} {'M2 (action gap)':<20} {'M2 (logprob gap)'}")
print("-" * 70)

for geo in ["lorentzian", "riemannian", "euclidean"]:
    path = f"./outputs/d0/d0_results_{geo}.json"
    if os.path.exists(path):
        with open(path) as f:
            r = json.load(f)
        print(f"{geo:<15} "
              f"{_fmt(r.get('m1_timelike_rate', 'N/A'), 18)} "
              f"{_fmt(r.get('m2_action_gap', 'N/A'), 20)} "
              f"{_fmt(r.get('m2_logprob_gap', 'N/A'))}")
    else:
        print(f"{geo:<15} (not run)")

In [ ]:
# Mostra i plot D0
import os, glob
from IPython.display import Image, display

for img in sorted(glob.glob("./outputs/d0/*.png")):
    print(f"\n--- {os.path.basename(img)} ---")
    display(Image(filename=img, width=600))

---
## 2. D1 — Synthetic Branching (H3)

Le continuazioni alternative sono separate da intervalli space-like?
Solo Lorentzian può ottenere BOTH time-like within-branch AND space-like cross-branch.

In [ ]:
!python -m scripts.train --config configs/d1_branching.yaml --geometry lorentzian

In [ ]:
!python -m scripts.train --config configs/d1_branching.yaml --geometry riemannian

In [ ]:
!python -m scripts.train --config configs/d1_branching.yaml --geometry euclidean

In [ ]:
# Confronta risultati D1
import os, json

def _fmt(v, w=10, prec=4):
    if isinstance(v, (int, float)):
        return f"{v:<{w}.{prec}f}"
    return f"{str(v):<{w}}"

print("=" * 90)
print("D1 RESULTS — BRANCHING EXPERIMENT")
print("=" * 90)
print(f"{'Geometry':<15} {'M1':<10} {'M3':<10} {'M3p within':<14} {'M3p cross':<14} {'M3p joint':<14}")
print("-" * 77)

for geo in ["lorentzian", "riemannian", "euclidean"]:
    path = f"./outputs/d1/d1_results_{geo}.json"
    if os.path.exists(path):
        with open(path) as f:
            r = json.load(f)
        print(f"{geo:<15} "
              f"{_fmt(r.get('m1_timelike_rate', 'N/A'), 10)} "
              f"{_fmt(r.get('m3_branching_separation', 'N/A'), 10)} "
              f"{_fmt(r.get('m3p_within_timelike', 'N/A'), 14)} "
              f"{_fmt(r.get('m3p_cross_spacelike', 'N/A'), 14)} "
              f"{_fmt(r.get('m3p_joint', 'N/A'), 14)}")
    else:
        print(f"{geo:<15} (not run)")

In [ ]:
# Plot D1
import os, glob
from IPython.display import Image, display

for img in sorted(glob.glob("./outputs/d1/*.png")):
    print(f"\n--- {os.path.basename(img)} ---")
    display(Image(filename=img, width=600))

---
## 3. D2 — WikiText-103 Real Text (la tesi vera)

### Step 1: Pre-encode il corpus
Questo step calcola:
- Embeddings dei paragrafi (sentence-transformer)
- Log-prob LM per ogni transizione (gpt2-medium)

Il risultato viene cachato in `./cache/` — basta runnarlo una volta.

**Nota tempi**: la config `configs/d2_wikitext.yaml` limita a `max_articles: 5000`
(~10–15 min di scoring LM su T4). Per il corpus completo (~28k articoli, **ore** di
scoring su T4) setta `max_articles: null` nello YAML oppure passa `--max_articles`
con un valore esplicito dalla CLI.

In [ ]:
# Pre-encode WikiText-103 (embeddings + LM scores).
# Con max_articles=5000 dal config: ~10-15 min su T4.
# Per overridare il tetto, decommenta la seconda forma:
!python -m scripts.encode_corpus --config configs/d2_wikitext.yaml
# !python -m scripts.encode_corpus --config configs/d2_wikitext.yaml --max_articles 2000

In [ ]:
# Verifica che la cache esista (legge cache_dir dalla config)
import os, yaml
try:
    with open("configs/d2_wikitext.yaml") as f:
        _cfg = yaml.safe_load(f)
    cache_dir = _cfg.get("cache_dir", "./cache")
except FileNotFoundError:
    cache_dir = "./cache"

if os.path.exists(cache_dir):
    print(f"Cache dir: {cache_dir}")
    for fn in sorted(os.listdir(cache_dir)):
        size = os.path.getsize(os.path.join(cache_dir, fn))
        print(f"  {fn:<45} {size / 1e6:.1f} MB")
else:
    print(f"Cache non trovata in {cache_dir} — runna la cella precedente.")

### Step 2: Train D2 (Lorentzian)

In [ ]:
# Train D2 con geometria Lorentziana
!python -m scripts.train --config configs/d2_wikitext.yaml

### Step 3 (opzionale): Train D2 baselines

In [ ]:
# Riemannian baseline
!python -m scripts.train --config configs/d2_wikitext.yaml --geometry riemannian

In [ ]:
# Euclidean baseline
!python -m scripts.train --config configs/d2_wikitext.yaml --geometry euclidean

In [ ]:
# Confronta risultati D2
import os, json

def _fmt(v, w=10, prec=4):
    if isinstance(v, (int, float)):
        return f"{v:<{w}.{prec}f}"
    return f"{str(v):<{w}}"

print("=" * 80)
print("D2 RESULTS — WIKITEXT-103 REAL TEXT")
print("=" * 80)
print(f"{'Geometry':<15} {'M1':<10} {'M4 Jaccard':<14} {'M4 Prec':<12} {'M4 Recall':<12} {'M5 NLL':<10}")
print("-" * 73)

for geo in ["lorentzian", "riemannian", "euclidean"]:
    path = f"./outputs/d2/d2_results_{geo}.json"
    if os.path.exists(path):
        with open(path) as f:
            r = json.load(f)
        m1 = r.get("m1", r.get("m1_timelike_rate", "N/A"))
        m5 = r.get("m5", r.get("m5_nll", "N/A"))
        print(f"{geo:<15} "
              f"{_fmt(m1, 10)} "
              f"{_fmt(r.get('m4_jaccard', 'N/A'), 14)} "
              f"{_fmt(r.get('m4_precision', 'N/A'), 12)} "
              f"{_fmt(r.get('m4_recall', 'N/A'), 12)} "
              f"{_fmt(m5, 10)}")
    else:
        print(f"{geo:<15} (not run)")

In [ ]:
# Plot D2
import os, glob
from IPython.display import Image, display

for img in sorted(glob.glob("./outputs/d2/*.png")):
    print(f"\n--- {os.path.basename(img)} ---")
    display(Image(filename=img, width=600))

---
## 4. Alternativa rapida: run_baselines (D0 + D1 in un colpo)

Se vuoi runnare D0 e D1 con tutte e 3 le geometrie in automatico:

In [ ]:
# ALTERNATIVA — DECOMMENTA la riga sotto SOLO se NON hai già eseguito
# manualmente le celle D0 / D1 qui sopra. Esegue le 3 geometrie su D0 e D1
# in un colpo solo (~15-20 min su T4).
# !python -m scripts.run_baselines

---
## 5. Download risultati

Scarica output, checkpoint e plot sul tuo computer.

In [ ]:
# Comprimi tutto per download.
# Di default escludiamo `cache/` (centinaia di MB di embeddings + LM scores
# che puoi sempre rigenerare). Metti INCLUDE_CACHE=True se ti servono.
import os

INCLUDE_CACHE = False

candidates = ["outputs", "checkpoints"]
if INCLUDE_CACHE:
    candidates.append("cache")
to_pack = [d for d in candidates if os.path.isdir(d)]

if not to_pack:
    print("Niente da impacchettare: nessuna delle directory outputs/, checkpoints/ esiste.")
else:
    print(f"Impacchetto: {to_pack}")
    !tar czf results.tar.gz {" ".join(to_pack)}
    size_mb = os.path.getsize("results.tar.gz") / 1e6
    print(f"results.tar.gz: {size_mb:.1f} MB")
    try:
        from google.colab import files
        files.download("results.tar.gz")
    except ImportError:
        print("Non su Colab — scarica manualmente results.tar.gz")